# Phase 9.5 — MentalBERT Training on Google Colab (Complete 188-Participant DAIC-WOZ)

**Purpose:** Run Ritik's ORIGINAL `trainer_mentalbert_daic.py` on the complete 188-participant dataset in Colab (GPU).

**Rules preserved:** no code modification, no architecture change, every original hyperparameter kept
(`--mode supervised --epochs 3 --batch-size 8 --lr 2e-5 --val-split 0.1 --binarize --binarize-threshold 10.0`,
model `mental/mental-bert-base-uncased`). The only difference vs the local run is the execution device (GPU),
which the trainer selects automatically via its own default `cuda if torch.cuda.is_available() else cpu`.

**Run order:** Cell 1 → 2 → 3 → 4 → 5 → 6 → 7. Cells 3 (HF login) and 4 (file upload) require your manual input.

## Cell 1 — Confirm GPU runtime
Runtime ▸ Change runtime type ▸ Hardware accelerator = **GPU** (T4 is fine).

In [ ]:
import torch, subprocess
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout[:600])

## Cell 2 — Install pinned dependencies
`transformers==4.44.0` is the critical pin: the trainer does `from transformers import AdamW`, which was removed in transformers >= 4.46. Torch is already present in Colab (CUDA build) and is left untouched.

In [ ]:
!pip install -q "transformers==4.44.0" "scikit-learn>=1.3" "pandas>=2.0" "pyarrow>=12" "numpy<2"
import transformers; from transformers import AdamW  # must import cleanly
print('transformers', transformers.__version__, '| AdamW import OK')

## Cell 3 — Hugging Face login (MANUAL: gated model)
`mental/mental-bert-base-uncased` is gated. Accept the model terms once on huggingface.co, then paste a token
with **read** access below. This is the only credential step.

In [ ]:
from huggingface_hub import login
login()  # paste your HF read token when prompted

## Cell 4 — Upload the two package files (MANUAL)
Upload **exactly these two files** from `colab_package/`:
1. `trainer_mentalbert_daic.py` (original, unmodified)
2. `daic_records.parquet` (188 records)

In [ ]:
from google.colab import files
up = files.upload()
import os
assert 'trainer_mentalbert_daic.py' in up, 'missing trainer_mentalbert_daic.py'
assert 'daic_records.parquet' in up, 'missing daic_records.parquet'
import pandas as pd
df = pd.read_parquet('daic_records.parquet')
print('records:', len(df), '| cols:', list(df.columns))
print('pos(>10):', int((df.phq_score>10).sum()), '| neg:', int((df.phq_score<=10).sum()))
assert len(df) == 188, f'expected 188 records, got {len(df)}'

## Cell 5 — Run the ORIGINAL trainer (exact hyperparameters)
`--device` is omitted so the trainer uses its own default and auto-selects the Colab GPU. All model/training
hyperparameters are the Phase 8B values.

In [ ]:
!python trainer_mentalbert_daic.py \
  --mode supervised \
  --input ./daic_records.parquet \
  --epochs 3 \
  --batch-size 8 \
  --lr 2e-5 \
  --val-split 0.1 \
  --binarize \
  --binarize-threshold 10.0

## Cell 6 — Show the results (training_report.json)

In [ ]:
import json, os
with open('trainer_outputs/training_report.json') as f:
    rep = json.load(f)
print(json.dumps(rep['metrics'], indent=2))
print('\nArtifacts in trainer_outputs/:')
for root,_,fs in os.walk('trainer_outputs'):
    for fn in fs:
        p = os.path.join(root, fn)
        print(f'  {os.path.getsize(p):>12,d}  {p}')

## Cell 7 — Download results back to your machine
**7a** auto-downloads the small metric artifacts (needed for the Phase 9.5 report/audit).
**7b** copies the full `trainer_outputs/` (incl. the two ~438 MB `.pt` files) to Google Drive — more reliable than
browser download for large files.

In [ ]:
# 7a — small metric artifacts (fast browser download)
from google.colab import files
for f in ['trainer_outputs/training_report.json',
          'trainer_outputs/eval_preds.csv',
          'trainer_outputs/modality_ablation.json',
          'trainer_outputs/mentalbert_delta.receipt.json']:
    files.download(f)

In [ ]:
# 7b — full outputs (incl. 438 MB checkpoint + delta) to Google Drive
from google.colab import drive
drive.mount('/content/drive')
import shutil, os
dst = '/content/drive/MyDrive/Phase95_trainer_outputs'
if os.path.exists(dst): shutil.rmtree(dst)
shutil.copytree('trainer_outputs', dst)
print('copied trainer_outputs -> ', dst)